In [1]:
# ==========================================
# CELL 1: SETUP REPO AND LIBRARIES
# ==========================================
!git clone https://github.com/mlpc-ucsd/Patch-DM.git
%cd Patch-DM

# Install compatible legacy Lightning and required packages
!pip install -q pytorch-lightning==1.9.5 lmdb==1.2.1 lpips pytorch-fid einops
!pip install -q git+https://github.com/openai/CLIP.git

print("✅ Repository cloned and dependencies installed.")

Cloning into 'Patch-DM'...
remote: Enumerating objects: 93, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 93 (delta 21), reused 85 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (93/93), 1.06 MiB | 9.14 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/kaggle/working/Patch-DM
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 881.5/881.5 kB 13.0 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.5/829.5 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.8 MB/s eta 0:00:00
✅ Repository cloned and dependencies installed.


In [5]:
# ==========================================
# CELL 2: PREPARE SEMANTIC CODES & WEIGHTS
# ==========================================
import os
import torch

# 1. Paths for Semantic Codes
INPUT_SEMANTIC = "/kaggle/input/datasets/pragatirokade/semantic/semantic.pt"
FIXED_SEMANTIC = "/kaggle/working/semantic_fixed.pt"

# 2. Path for Stage 2 Weights (UPDATE THIS!)
STAGE2_WEIGHTS_INPUT = "/kaggle/input/datasets/pragatirokade/weights/last.ckpt" 

if not os.path.exists(STAGE2_WEIGHTS_INPUT):
    print("⚠️ WARNING: You have not linked your Stage 2 weights. The training cell will fail.")

# 3. Re-key the semantic dictionary if necessary
if os.path.exists(INPUT_SEMANTIC):
    data = torch.load(INPUT_SEMANTIC, map_location='cpu')
    
    if torch.is_tensor(data):
        print("📦 Input is a raw tensor. Wrapping for Patch-DM...")
        wrapped = {'model_state_dict': {'semantic_enc.weight': data}}
        torch.save(wrapped, FIXED_SEMANTIC)
    elif isinstance(data, dict) and 'model_state_dict' in data:
        print("✅ Input is already in the correct dictionary format.")
        torch.save(data, FIXED_SEMANTIC)
    else:
        print("🔧 Re-keying dictionary for training...")
        key = list(data.keys())[0] if isinstance(data, dict) else None
        tensor = data[key] if key else data
        wrapped = {'model_state_dict': {'semantic_enc.weight': tensor}}
        torch.save(wrapped, FIXED_SEMANTIC)
        
    print(f"🚀 Semantic file ready at: {FIXED_SEMANTIC}")
else:
    print("❌ Could not find semantic.pt in Input. Check dataset attachment!")

✅ Input is already in the correct dictionary format.
🚀 Semantic file ready at: /kaggle/working/semantic_fixed.pt


In [8]:
# ==========================================
# CELL 3: PATCH CODE FOR KAGGLE COMPATIBILITY
# ==========================================
import os
import re

# Wipe the broken files and restore them to the original GitHub state
os.system('git checkout experiment.py train_latent.py templates_latent.py')

# 1. Fix experiment.py (Numpy, Hooks, Heartbeat, and Trainer Timeouts)
with open('experiment.py', 'r') as f:
    content = f.read()

content = content.replace('from numpy.lib.function_base import flip', 'from numpy import flip')

# Inject Heartbeat and fix hook signature
old_hook = r'def on_train_batch_end\s*\([^)]+\)\s*->\s*None:'
new_hook = '''def on_train_batch_end(self, outputs, batch, batch_idx: int, dataloader_idx: int = 0, **kwargs) -> None:
        if self.global_step % 100 == 0:
            print(f"⏳ HEARTBEAT | Step: {self.global_step} | Latent Training is ACTIVE.")
'''
content = re.sub(old_hook, new_hook, content)

content = content.replace("desc='infer')", "desc='infer', disable=True)")
content = content.replace(
    'trainer = pl.Trainer(', 
    'trainer = pl.Trainer(enable_progress_bar=False, max_time="00:11:30:00", '
)
with open('experiment.py', 'w') as f:
    f.write(content)

# 2. Patch train_latent.py to accept the Semantic Dataset
with open('train_latent.py', 'r') as f:
    latent_content = f.read()

# Point directly to the fixed semantic file generated in Cell 2
latent_content = latent_content.replace(
    'conf.sample_size = 1', 
    'conf.sample_size = 1\n    conf.data_path = "/kaggle/working/semantic_fixed.pt"'
)
with open('train_latent.py', 'w') as f:
    f.write(latent_content)

# 3. Increase checkpoint frequency in templates_latent.py
with open('templates_latent.py', 'r') as f:
    templates_content = f.read()
# Change from saving every 2_000_000 samples to every 500_000 samples
templates_content = templates_content.replace('conf.save_every_samples = 2_000_000', 'conf.save_every_samples = 500_000')
with open('templates_latent.py', 'w') as f:
    f.write(templates_content)

print("✅ Headless optimizations, heartbeat, and Semantic dataset path injected!")

✅ Headless optimizations, heartbeat, and Semantic dataset path injected!


Updated 3 paths from the index


In [9]:
# ==========================================
# CELL 4: LAUNCH TRAINING WITH DISK GUARDIAN
# ==========================================
import subprocess
import time
import os

# 1. Verify we are in the right directory
if not os.path.exists('train_latent.py'):
    %cd /kaggle/working/Patch-DM/

STAGE2_WEIGHTS = "/kaggle/input/datasets/pragatirokade/weights/last.ckpt" 
CHECKPOINT_DIR = '/kaggle/working/Patch-DM/checkpoints/latent_ffhq/'

if not os.path.exists(STAGE2_WEIGHTS):
    raise FileNotFoundError("❌ Please update STAGE2_WEIGHTS with your uploaded Kaggle dataset path!")

# 2. Start Latent Training (Using -u for unbuffered logs)
print("🔥 Launching Latent Training (Unbuffered Logs)...")
train_cmd = [
    "python", "-u", "train_latent.py",
    "--name", "latent_ffhq",
    "--model_path", STAGE2_WEIGHTS
]

# Redirect stdout and stderr to the notebook's output
process = subprocess.Popen(train_cmd, stdout=None, stderr=None)

# 3. Disk Guardian
print("🛡️ Disk Guardian Active (Watching Latent Checkpoints)...")
try:
    while process.poll() is None:
        if os.path.exists(CHECKPOINT_DIR):
            files = [os.path.join(CHECKPOINT_DIR, f) for f in os.listdir(CHECKPOINT_DIR) if f.endswith('.ckpt')]
            if len(files) > 2:
                files.sort(key=os.path.getmtime)
                for f in files[:-2]: 
                    os.remove(f)
                    print(f"🗑️ Cleaned old checkpoint: {os.path.basename(f)}")
        time.sleep(300) # Check every 5 minutes
    
    if process.returncode != 0:
        print(f"❌ Training stopped unexpectedly with code {process.returncode}")
    else:
        print("✅ Training completed or timed out gracefully!")
except KeyboardInterrupt:
    process.terminate()
    print("🛑 Process terminated by user.")

🔥 Launching Latent Training (Unbuffered Logs)...
🛡️ Disk Guardian Active (Watching Latent Checkpoints)...
conf: latent_ffhq


Global seed set to 0


Model params: 119.70 M
==Model size is 64==
loading pretrain ... 72M
step: 135426


ModelCheckpoint(save_last=True, save_top_k=-1, monitor=None) will duplicate the last checkpoint saved.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/connectors/accelerator_connector.py:478: LightningDeprecationWarning: Setting `Trainer(gpus=[0])` is deprecated in v1.7 and will be removed in v2.0. Please use `Trainer(accelerator='gpu', devices=[0])` instead.
  rank_zero_deprecation(
Using 16bit None Automatic Mixed Precision (AMP)
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/plugins/precision/native_amp.py:47: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


ckpt path: checkpoints/latent_ffhq/last.ckpt
local seed: 0


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name      | Type                 | Params
---------------------------------------------------
0 | model     | BeatGANsAutoencModel | 125 M 
1 | ema_model | BeatGANsAutoencModel | 125 M 
---------------------------------------------------
125 M     Trainable params
125 M     Non-trainable params
251 M     Total params
502.055   Total estimated model params size (MB)
2026-03-26 12:40:25.035918: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774528825.272009     205 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774528825.335707     205 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:17745288

on train dataloader start ...
mean: tensor(0.0280, device='cuda:0') std: tensor(0.2727, device='cuda:0')


/kaggle/working/Patch-DM/experiment.py:350: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(False):
/kaggle/working/Patch-DM/diffusion/base.py:254: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(self.conf.fp16):


⏳ HEARTBEAT | Step: 100 | Latent Training is ACTIVE.
⏳ HEARTBEAT | Step: 200 | Latent Training is ACTIVE.
⏳ HEARTBEAT | Step: 300 | Latent Training is ACTIVE.
⏳ HEARTBEAT | Step: 400 | Latent Training is ACTIVE.
⏳ HEARTBEAT | Step: 500 | Latent Training is ACTIVE.
⏳ HEARTBEAT | Step: 600 | Latent Training is ACTIVE.
⏳ HEARTBEAT | Step: 700 | Latent Training is ACTIVE.
⏳ HEARTBEAT | Step: 800 | Latent Training is ACTIVE.
⏳ HEARTBEAT | Step: 900 | Latent Training is ACTIVE.
⏳ HEARTBEAT | Step: 1000 | Latent Training is ACTIVE.
⏳ HEARTBEAT | Step: 1100 | Latent Training is ACTIVE.
⏳ HEARTBEAT | Step: 1200 | Latent Training is ACTIVE.
⏳ HEARTBEAT | Step: 1300 | Latent Training is ACTIVE.
⏳ HEARTBEAT | Step: 1400 | Latent Training is ACTIVE.
⏳ HEARTBEAT | Step: 1500 | Latent Training is ACTIVE.
⏳ HEARTBEAT | Step: 1600 | Latent Training is ACTIVE.
⏳ HEARTBEAT | Step: 1700 | Latent Training is ACTIVE.
⏳ HEARTBEAT | Step: 1800 | Latent Training is ACTIVE.
⏳ HEARTBEAT | Step: 1900 | Latent Tra

In [ ]:
# ==========================================
# CELL 5: FINAL BACKUP
# ==========================================
import time
print("📦 Zipping results...")
%cd /kaggle/working/
time.sleep(5)
!zip -r /kaggle/working/latent_weights_backup.zip Patch-DM/checkpoints/latent_ffhq/
print("✅ Backup complete. Look for 'latent_weights_backup.zip' in your output folder.")